# Chapter 18: Fine-tuning your own model

ShopFlow ticket triage (Chapter 2's task), fine-tuned into Qwen3-1.7B and Qwen3-4B with LoRA and
QLoRA on one RTX 3090, served with vLLM `--enable-lora`, and compared with prompted models.

**How this notebook relates to the GPU work.** Training and serving need the chapter's CUDA venv
(`.venv-cuda`, see `requirements-cuda.txt`) and the whole GPU, so they ran as scripts:
`ch18_train.py` (Unsloth / PEFT training), `serve_vllm.sh` (vLLM on port 8218) and `ch18_eval.py`
(the evaluation client). Each script wrote its measurements to `results/*.json`. This notebook
**reads those JSON files** and computes every table and chart in the README from them. It needs
neither the GPU nor a model server, so it runs headless in the main venv.

In [1]:
import collections, json, re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from ch18_data import (DATA, FORMAT_HINT, FT_RULES, answer_json, load_templates, read_jsonl,
                       to_example)
from ch18_eval import local_cost_per_1k

RESULTS = Path("results")
IMAGES = Path("../images")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
TEMPLATES = load_templates()
split = {n: read_jsonl(DATA / f"{n}.jsonl") for n in ("train", "val", "test", "synthetic")}
{n: len(rows) for n, rows in split.items()}

{'train': 844, 'val': 150, 'test': 210, 'synthetic': 290}

## The data

1,500 templated tickets are split 1,200 / 150 / 150 (seed 18). The 60 hand-written hard tickets
go to the test set only, so the test set has 210 tickets and the model never trains on anything
that looks like them.

Then we remove the leak Chapter 19 found: templates with no variable field produce identical
tickets, so 43 test and 41 val tickets had exact copies in train. `ch18_prepare.py dedupe` drops
every training row whose text (stripped, lower-cased) also appears in val or test. Eight templates
consist *only* of identical text; for each we keep exactly one row, so the model still learns the
intent, and their test tickets form a separate "identical text" segment.

In [2]:
dd = json.loads((RESULTS / "dedupe.json").read_text())
print(f"train: {dd['train_before']} rows, {dd['dropped']} dropped, {dd['restored']} restored "
      f"-> {dd['train_after']}")
print("identical-text templates (one training row kept each):")
for i, s in dd["identical_text_templates"].items():
    print(f"  {i:>2} {s}")
segs = json.loads((RESULTS / "test_segments.json").read_text())
print("test segments:", segs["counts"])
tr_keys = [(r["subject"].strip().lower(), r["body"].strip().lower()) for r in split["train"]]
print(f"exact duplicates still inside train: {len(tr_keys) - len(set(tr_keys))} of {len(tr_keys)} rows")

train: 1200 rows, 364 dropped, 8 restored -> 844
identical-text templates (one training row kept each):
   8 What is your return policy?
  16 Change my email address
  17 Someone else used my account
  18 Delete my account
  19 Unsubscribe from marketing emails
  21 Payment keeps failing
  22 Do you accept PayPal?
  27 Store opening hours
test segments: {'unseen_templated': 113, 'identical_text': 37, 'hard': 60}
exact duplicates still inside train: 20 of 844 rows


In [3]:
rows = []
for name in ("train", "val", "test"):
    df = pd.DataFrame(split[name])
    rows.append({"split": name, "tickets": len(df), "hard": int(df.hard.sum()),
                 "templates seen": df.template_id.dropna().nunique(),
                 **df.category.value_counts().to_dict()})
print(pd.DataFrame(rows).fillna(0).to_string(index=False))

split  tickets  hard  templates seen  product_defect  shipping  refund  payment  other  account
train      844     0              30             185       185     152      138    137       47
  val      150     0              29              36        21      30       26     23       14
 test      210    60              30              34        46      34       37     25       34


In [4]:
# One training row as it sits in data/train.jsonl, and the chat example built from it.
r = split["train"][0]
print(json.dumps(r, indent=1, ensure_ascii=False))
for m in to_example(r)["messages"]:
    print(f"--- {m['role']}\n{m['content']}")

{
 "id": 1,
 "subject": "Unsubscribe from marketing emails",
 "body": "I keep getting promo emails even though I unsubscribed. Please remove me from the list.",
 "channel": "email",
 "category": "account",
 "priority": "low",
 "order_id": null,
 "hard": false,
 "template_id": 19,
 "source": "shopflow",
 "sentiment": "negative",
 "summary": "The customer continues receiving promotional emails after unsubscribing and asks to be removed from the mailing list.",
 "text_labels_by": "openai/gpt-6-luna"
}
--- system
You triage customer-support tickets for ShopFlow, an online shop.

Reply with a single JSON object with exactly these keys: category, priority, sentiment, order_id (integer or null), summary. Reply with JSON only.
--- user
<ticket>
channel: email
subject: Unsubscribe from marketing emails
body: I keep getting promo emails even though I unsubscribed. Please remove me from the list.
</ticket>
--- assistant
{"category": "account", "priority": "low", "sentiment": "negative", "order_id

### Teacher labels: gpt-6-luna fills in sentiment and summary

The table has gold `category`, `priority` and `order_id`, but no `sentiment` or `summary`. We let
`openai/gpt-6-luna` triage every train and val ticket with Chapter 2's prompt, keep its
`sentiment` and `summary`, and keep the **gold** routing labels. How often did the teacher agree
with the gold routing labels?

In [5]:
teacher = {}
for name in ("train", "val"):
    for d in json.loads((RESULTS / f"teacher__{name}.json").read_text()):
        teacher[d["ticket_id"]] = d
tr = split["train"] + split["val"]
agree = {f: round(100 * sum(teacher[r["id"]]["triage"][f] == r[f] for r in tr) / len(tr), 1)
         for f in ("category", "priority", "order_id")}
cost = sum(d["cost"] for d in teacher.values())
print(f"{len(tr)} tickets, teacher agreement with gold: {agree}; teacher cost ${cost:.4f}")

994 tickets, teacher agreement with gold: {'category': 85.8, 'priority': 62.9, 'order_id': 100.0}; teacher cost $0.1227


In [6]:
# Where the teacher disagrees, it disagrees on whole templates: a policy difference, not noise.
per = collections.defaultdict(collections.Counter)
for r in tr:
    t = teacher[r["id"]]["triage"]
    per[r["template_id"]][(t["category"], t["priority"])] += 1
out = []
for i, tp in enumerate(TEMPLATES):
    gold_p = "/".join(tp["priority"]) if isinstance(tp["priority"], list) else tp["priority"]
    (tc, tpr), n = per[i].most_common(1)[0]
    if tc != tp["category"] or tpr not in gold_p.split("/"):
        out.append({"template": tp["subject"][:34], "gold": f"{tp['category']}/{gold_p}",
                    "gpt-6-luna says (most often)": f"{tc}/{tpr}",
                    "share": f"{n}/{sum(per[i].values())}"})
print(pd.DataFrame(out).to_string(index=False))
print(f"\n{len(out)} of {len(TEMPLATES)} templates: the teacher's usual answer differs from ShopFlow's label")

                          template                  gold gpt-6-luna says (most often) share
    Where is my order #{order_id}?   shipping/low/medium                shipping/high 27/42
Need order #{order_id} before Frid         shipping/high              shipping/medium 39/39
           Refund not received yet         refund/medium                  refund/high 40/54
               Item arrived broken   product_defect/high        product_defect/medium 40/40
             Wrong colour received product_defect/medium              shipping/medium 44/44
Question about {product} materials    product_defect/low                    other/low 39/39
                      Can't log in        account/medium                 account/high 35/43
      Someone else used my account          account/high               account/urgent   4/4
                 Delete my account           account/low               account/medium   8/8
 Unsubscribe from marketing emails           account/low                    othe

### The ceiling: how much priority accuracy is even possible?

Four templates draw their priority at random from two values (for example "Where is my order?" is
`low` or `medium` regardless of the text). No model can beat a coin flip on those. The best any
model can do is always predict the more frequent value of each such template.

In [7]:
# The best a model can do: learn each template's most frequent priority from the training set.
by_t = collections.defaultdict(collections.Counter)
for r in split["train"]:
    by_t[r["template_id"]][r["priority"]] += 1
majority = {t: c.most_common(1)[0][0] for t, c in by_t.items()}

test_tpl = [r for r in split["test"] if not r["hard"]]
noisy = [i for i, tp in enumerate(TEMPLATES) if isinstance(tp["priority"], list)]
print("templates with a random priority:", [TEMPLATES[i]["subject"][:30] for i in noisy])
print("train counts:", {TEMPLATES[i]["subject"][:20]: dict(by_t[i]) for i in noisy})
CEILING = {}
for seg in ("unseen_templated", "identical_text"):
    rs = [r for r in test_tpl if segs["segment"][str(r["id"])] == seg]
    hit = sum(majority.get(r["template_id"], r["priority"]) == r["priority"] for r in rs)
    CEILING[seg] = round(100 * hit / len(rs), 1)
    print(f"{seg}: {len(rs)} tickets, {sum(r['template_id'] in noisy for r in rs)} from those "
          f"templates, estimated priority ceiling {CEILING[seg]}%")

templates with a random priority: ['Where is my order #{order_id}?', 'Return request for order #{ord', '{product} stopped working', 'Discount code not applied']
train counts: {'Where is my order #{': {'medium': 21, 'low': 18}, 'Return request for o': {'low': 16, 'medium': 9}, '{product} stopped wo': {'medium': 22, 'high': 20}, 'Discount code not ap': {'medium': 29, 'high': 14}}
unseen_templated: 113 tickets, 20 from those templates, estimated priority ceiling 93.8%
identical_text: 37 tickets, 0 from those templates, estimated priority ceiling 100.0%


This is an **estimate**, not a hard bound: it assumes a model can only learn each random
template's most frequent training value. With 113 unseen tickets, a model that happens to guess
a few of the coin flips right lands above it, and the 1.7B LoRA does (97.3%).

### Synthetic tickets for v2

`ch18_prepare.py synth` asked gpt-6-luna for 300 messy tickets (sarcasm, several issues, other
languages, forwarded threads, injection attempts...) with requested labels. A second, independent
triage pass (which never saw the requested labels) had to agree on the category, and nothing may
be too similar to a hard test ticket.

In [8]:
syn = pd.DataFrame(split["synthetic"])
raw = json.loads((RESULTS / "synth__raw.json").read_text())
qa = json.loads((RESULTS / "teacher__synth_qa.json").read_text())
print(f"{len(qa)} generated, {len(syn)} kept after the QA pass; cost "
      f"${sum(j['cost'] or 0 for j in raw) + sum(d['cost'] for d in qa):.4f}")
print(syn["style"].value_counts().to_string())
print("\nmax word-trigram Jaccard to any hard test ticket:",
      syn.max_sim_to_hard_test.describe()[["50%", "max"]].round(3).to_dict())
for r in split["synthetic"][:: max(1, len(syn) // 4)][:4]:
    print(f"\n[{r['style']}] {r['category']}/{r['priority']} order_id={r['order_id']}\n  {r['subject']}: {r['body'][:220]!r}")

300 generated, 290 kept after the QA pass; cost $0.0639
style
sarcastic                                   20
written in Spanish                          20
written in French                           20
written in Indonesian                       20
written in Portuguese                       20
angry, partly in capital letters            20
contains an instruction aimed at the AI     20
several issues in one ticket                19
written in Italian                          19
very short and vague, a few words, typos    19
long and rambling, with irrelevant perso    19
polite and formal business customer         19
mixes two languages (English plus anothe    19
written in German                           18
forwarded email thread with quoted repli    18

max word-trigram Jaccard to any hard test ticket: {'50%': 0.0, 'max': 0.075}

[sarcastic] shipping/medium order_id=6939
  What a lovely tracking update: 'Fantastic, the tracking for order #6939 hasn’t changed in two days, even thoug

## Training runs

`ch18_train.py` wrote one `results/train__<name>.json` per run: wall-clock training time, peak
VRAM as PyTorch saw it, trainable parameters, and the loss curves.

In [9]:
train_runs = {}
for p in sorted(RESULTS.glob("train__*.json")):
    r = json.loads(p.read_text())
    train_runs[r["name"]] = r
tt = pd.DataFrame([{
    "run": n, "backend": r["backend"], "base": r["model"].split("/")[-1], "frozen base": r["quant"],
    "data": "+".join(f.removesuffix(".jsonl") for f in r["train_files"]), "examples": r["n_train"],
    "steps": r["steps"], "train s": r["train_seconds"], "load s": r["load_seconds"],
    "peak VRAM GB": r["peak_vram_reserved_gb"], "trainable M": round(r["trainable_params"] / 1e6, 1),
    "eval loss": r["final_eval_loss"], "adapter MB": r["adapter_mb"],
} for n, r in train_runs.items()])
print(tt.to_string(index=False))
first = train_runs[tt.run.iloc[0]]
print("\nversions:", first["versions"], "| GPU:", first["gpu"])

           run backend       base frozen base            data  examples  steps  train s  load s  peak VRAM GB  trainable M  eval loss  adapter MB
 q17-lora-peft    peft Qwen3-1.7B        bf16           train       844    106    113.8    12.1          4.16         17.4     0.1040        66.5
   q17-lora-v2 unsloth Qwen3-1.7B        bf16 train+synthetic      1134    142     89.6    47.4          3.61         17.4     0.0998        66.5
      q17-lora unsloth Qwen3-1.7B        bf16           train       844    106     66.0    40.8          3.61         17.4     0.1007        66.5
q17-qlora-peft    peft Qwen3-1.7B    4bit-nf4           train       844    106    113.2    10.4          2.22         17.4     0.1131        33.3
     q17-qlora unsloth Qwen3-1.7B    4bit-nf4           train       844    106     70.0    41.0          1.65         17.4     0.1091        66.5
    q4-lora-v2 unsloth   Qwen3-4B        bf16 train+synthetic      1134    142    207.2    45.5          8.10         33.0  

In [10]:
# Exactly the tokens that carried loss in the first training example (from the training log).
ex = train_runs["q17-lora"]["example_0"]
print(ex["prompt"][-120:])
print("--- in the loss:", repr(ex["tokens_in_loss"]))

ven though I unsubscribed. Please remove me from the list.
</ticket><|im_end|>
<|im_start|>assistant
<think>

</think>


--- in the loss: '{"category": "account", "priority": "low", "sentiment": "negative", "order_id": null, "summary": "The customer continues receiving promotional emails after unsubscribing and asks to be removed from the mailing list."}<|im_end|>'


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, color in (("q17-lora", "#5b8fc7"), ("q17-lora-v2", "#4f9a4f"), ("q4-lora", "#c98b2b")):
    if name not in train_runs:
        continue
    r = train_runs[name]
    axes[0].plot(*zip(*r["loss"]), color=color, alpha=0.6, label=f"{name} train")
    axes[0].plot(*zip(*r["eval_loss"]), color=color, marker="o", ls="--", label=f"{name} val")
axes[0].set(xlabel="optimizer step", ylabel="loss on answer tokens (log)", yscale="log",
            title="Loss curves")
axes[0].legend(fontsize=8)
cmp = tt.set_index("run")
axes[1].barh(cmp.index, cmp["train s"], color="#5b8fc7")
for i, (s, v) in enumerate(zip(cmp["train s"], cmp["peak VRAM GB"])):
    axes[1].text(s, i, f"  {s:.0f} s, {v:.1f} GB", va="center", fontsize=8)
axes[1].invert_yaxis()
axes[1].set(xlabel="training time in seconds (label: time, peak VRAM)", title="Cost of each run")
axes[1].set_xlim(0, cmp["train s"].max() * 1.45)
plt.tight_layout()
plt.savefig(IMAGES / "18-training.png", dpi=150)
plt.show()

## Evaluation: 210 test tickets, scored by segment

After the dedupe, 37 of the 150 templated test tickets still have the same text as a training row
(the eight identical-text templates; segments are defined exactly as in Chapter 19's
`evaluate.segments`). A fine-tuned model can memorise those, so the **headline** is reported as
two columns on the 173 tickets nobody trained on: 113 *unseen* templated tickets and the 60
*hard* ones. *Identical text* (37) is shown separately, as a memorisation score.

`ch18_eval.py` ran every system on the same tickets through Chapter 2's `triage_one` with strict
`json_schema`. Latency (per ticket at concurrency 1), validity and hosted cost are measured on the
173 headline tickets. For hosted models, $/1k is OpenRouter's billed `usage.cost`. For
self-hosted models, $/1k is the GPU's rental price divided by the
measured throughput with many requests in flight (concurrency 32 on vLLM, 4 on Ollama), with an
RTX 3090 at $0.50/h (RunPod, as of 2026-11).

In [12]:
GPU_USD_PER_HOUR = 0.50
SYSTEMS = [
    ("gpt-6-luna, Ch. 2 prompt", "openrouter__openai_gpt-6-luna__ch02"),
    ("gpt-6-luna, + 22 examples", "openrouter__openai_gpt-6-luna__fewshot"),
    ("gpt-oss:20b, Ch. 2 prompt", "ollama__gpt-oss_20b__ch02"),
    ("Qwen3-1.7B, Ch. 2 prompt", "vllm__Qwen_Qwen3-1.7B__ch02"),
    ("Qwen3-1.7B, + 22 examples", "vllm__Qwen_Qwen3-1.7B__fewshot"),
    ("Qwen3-4B, Ch. 2 prompt", "vllm__Qwen_Qwen3-4B__ch02"),
    ("Qwen3-4B, + 22 examples", "vllm__Qwen_Qwen3-4B__fewshot"),
    ("1.7B LoRA v1", "vllm__q17-lora__ft"),
    ("1.7B QLoRA v1", "vllm__q17-qlora__ft"),
    ("1.7B LoRA v1 (plain PEFT)", "vllm__q17-lora-peft__ft"),
    ("1.7B LoRA v2 (+synthetic)", "vllm__q17-lora-v2__ft"),
    ("4B LoRA v1", "vllm__q4-lora__ft"),
    ("4B QLoRA v1", "vllm__q4-qlora__ft"),
    ("4B LoRA v2 (+synthetic)", "vllm__q4-lora-v2__ft"),
]


def load(tag, c):
    p = RESULTS / f"eval__{tag}__c{c}.json"
    return json.loads(p.read_text()) if p.exists() else None


rows, EVALS = [], {}
for label, tag in SYSTEMS:
    one = load(tag, 1)
    if one is None:  # e.g. gpt-oss:20b before its GPU run
        continue
    EVALS[label] = one
    s = one["scores"]  # latency, cost and validity on the 173 headline tickets
    busy = load(tag, 32) or load(tag, 4)
    if one["endpoint"] == "openrouter":
        cost = s["headline"]["cost_per_1k_usd"]
    else:
        cost = local_cost_per_1k((busy or one)["tickets_per_second"], GPU_USD_PER_HOUR)
    rows.append({
        "system": label, "valid %": s["headline"]["final_valid_%"],
        "1st try %": s["headline"]["first_try_valid_%"],
        "cat unseen": s["unseen"]["category_acc_%"], "cat hard": s["hard"]["category_acc_%"],
        "cat identical": s["seen"]["category_acc_%"],
        "pri unseen": s["unseen"]["priority_acc_%"], "pri hard": s["hard"]["priority_acc_%"],
        "pri identical": s["seen"]["priority_acc_%"],
        "order_id %": s["headline"]["order_id_acc_%"],
        "p50 ms": s["headline"]["latency_p50_ms"], "p95 ms": s["headline"]["latency_p95_ms"],
        "busy tickets/s": busy["tickets_per_second"] if busy else None,
        "$ / 1k": round(cost, 4),
    })
RES = pd.DataFrame(rows)
print(RES.to_string(index=False))
print(f"\nestimated priority ceiling (label noise): {CEILING}")

                   system  valid %  1st try %  cat unseen  cat hard  cat identical  pri unseen  pri hard  pri identical  order_id %  p50 ms  p95 ms  busy tickets/s  $ / 1k
 gpt-6-luna, Ch. 2 prompt    100.0      100.0        83.2      91.7           89.2        66.4      65.0           62.2       100.0    2838    3786             NaN  0.0961
gpt-6-luna, + 22 examples     99.4       99.4       100.0      96.7          100.0        93.8      70.0           54.1        99.4    2764    3978             NaN  0.0744
gpt-oss:20b, Ch. 2 prompt    100.0       82.1        79.6      86.7           86.5        50.4      48.3           62.2       100.0     658    1269           2.469  0.0563
 Qwen3-1.7B, Ch. 2 prompt    100.0      100.0        84.1      80.0           91.9        38.1      31.7           45.9        88.4     310     355          54.416  0.0026
Qwen3-1.7B, + 22 examples    100.0      100.0        99.1      85.0           89.2        92.0      41.7           37.8        99.4     299 

In [13]:
fig, ax = plt.subplots(figsize=(10, 6))
groups = []  # points at the same accuracy and a similar cost share one label
for _, r in RES.sort_values(["cat hard", "$ / 1k"]).iterrows():
    hosted = r["system"].startswith("gpt-6")
    ft = "LoRA" in r["system"]
    ax.scatter(r["$ / 1k"], r["cat hard"], s=80, marker="s" if hosted else ("o" if ft else "^"),
               color="#c98b2b" if hosted else ("#4f9a4f" if ft else "#5b8fc7"))
    g = groups[-1] if groups else None
    if g and g["y"] == r["cat hard"] and r["$ / 1k"] / g["x"] < 1.5:
        g["labels"].append(r["system"])
    else:
        groups.append({"x": r["$ / 1k"], "y": r["cat hard"], "labels": [r["system"]]})
for g in groups:
    ax.annotate("\n".join(g["labels"]), (g["x"], g["y"]), fontsize=7, xytext=(7, -3),
                textcoords="offset points", va="top")
ax.set(xscale="log", xlabel="$ per 1,000 tickets (log scale)",
       ylabel="category accuracy on the 60 hard tickets (%)",
       title="Hosted (square), prompted local (triangle), fine-tuned local (circle)")
ax.grid(alpha=0.3)
ax.set_ylim(RES["cat hard"].min() - 3, 100)
plt.tight_layout()
plt.savefig(IMAGES / "18-accuracy-vs-cost.png", dpi=150)
plt.show()

In [14]:
keys = [k for k in ("gpt-6-luna, Ch. 2 prompt", "gpt-6-luna, + 22 examples",
                    "Qwen3-1.7B, Ch. 2 prompt", "1.7B LoRA v1", "1.7B LoRA v2 (+synthetic)",
                    "4B LoRA v2 (+synthetic)") if k in set(RES.system)]
sub = RES.set_index("system").loc[keys]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, (field, title) in zip(axes, (("cat", "Category"), ("pri", "Priority"))):
    sub[[f"{field} unseen", f"{field} hard", f"{field} identical"]].plot.bar(
        ax=ax, color=["#5b8fc7", "#c98b2b", "#bbbbbb"], rot=20)
    ax.set(title=f"{title} accuracy by segment", ylabel="%", xlabel="")
    ax.legend(["unseen templated (113)", "hard (60)", "identical text (37)"], fontsize=8,
              loc="lower left")
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
c_un = CEILING["unseen_templated"]
axes[1].axhline(c_un, ls="--", color="#888")
axes[1].text(len(keys) - 0.5, c_un + 1, f"estimated ceiling, unseen {c_un}%", fontsize=8,
             color="#555", ha="right")
plt.tight_layout()
plt.savefig(IMAGES / "18-segments.png", dpi=150)
plt.show()

### First-attempt failures

`valid %` counts a reply that became a valid `TicketTriage` after Chapter 2's retry (the model is
shown its error and asked again). `1st try %` counts only first attempts. A first reply that hit
`max_tokens` is counted as `truncated` and not retried; `no_output` means the reply finished
(`finish_reason` other than `length`) with empty content.

In [15]:
for label, e in EVALS.items():
    errs = e["scores"]["all"]["errors"]
    if errs:
        rs = e["results"]
        retried = [r for r in rs if r["attempts"] > 1]
        print(f"{label}: first-attempt errors {errs}; {len(retried)} of {len(rs)} tickets "
              f"retried, {sum(r['triage'] is not None for r in retried)} recovered")
        for r in [r for r in rs if r["errors"] and r["errors"][0].startswith("not_json")][:2]:
            print(f"  #{r['ticket_id']} {r['errors'][0][:110]}")

gpt-6-luna, + 22 examples: first-attempt errors {'truncated': 1}; 0 of 210 tickets retried, 0 recovered
gpt-oss:20b, Ch. 2 prompt: first-attempt errors {'no_output': 25, 'not_json': 11}; 36 of 210 tickets retried, 36 recovered
  #372 not_json: no JSON object found in reply: 'category: payment  \npriority: low  \nsentiment: neutral  \norder_id
  #621 not_json: no JSON object found in reply: 'category: product_defect  \npriority: medium  \nsentiment: negative 
Qwen3-4B, + 22 examples: first-attempt errors {'truncated': 1}; 0 of 210 tickets retried, 0 recovered


### Where the fine-tuned models still fail

In [16]:
test_by_id = {r["id"]: r for r in split["test"]}


def misses(label, hard=True):
    out = []
    for r in EVALS[label]["results"]:
        t = test_by_id[r["ticket_id"]]
        if t["hard"] == hard and r["triage"] and r["triage"]["category"] != t["category"]:
            out.append((t, r["triage"]))
    return out


for label in [k for k in ("1.7B LoRA v1", "1.7B LoRA v2 (+synthetic)") if k in EVALS]:
    m = misses(label)
    print(f"== {label}: {len(m)} hard tickets with the wrong category")
    for t, p in m[:8]:
        print(f"#{t['id']} gold {t['category']}/{t['priority']} -> {p['category']}/{p['priority']}"
              f" | {t['subject']}: {t['body'][:100]!r}")

== 1.7B LoRA v1: 13 hard tickets with the wrong category
#1501 gold shipping/medium -> other/low | Wow, amazing service: "Oh fantastic, my order #2943 is 'out for delivery' for the 9th day in a row. Truly world-class logis"
#1505 gold account/medium -> other/medium | Great job team: "Great job, your password reset email arrives exactly never. I've tried 6 times."
#1508 gold account/high -> other/medium | Absolutely thrilled: 'Absolutely thrilled that my account got locked right before Black Friday. Can someone actually unloc'
#1523 gold refund/high -> payment/medium | Rückerstattung fehlt: 'Ich habe die Bestellung #3510 vor drei Wochen zurückgeschickt und noch kein Geld erhalten.'
#1537 gold account/high -> other/medium | GDPR request: 'Under GDPR Article 15 I request a copy of all personal data you hold about me within 30 days.'
#1538 gold shipping/medium -> product_defect/medium | Wrong item: 'I received a yoga mat. I ordered a cast iron skillet. Order #13356. Honestly, kind of funny

## Spend

Every OpenRouter call in this chapter, summed from the cached replies (`usage.cost`).

In [17]:
spend = {f"teacher labels ({len(teacher):,} tickets)": sum(d["cost"] for d in teacher.values()),
         "synthetic generation + QA pass": sum(j["cost"] or 0 for j in raw)
         + sum(d["cost"] for d in qa)}
for label, e in EVALS.items():
    if e["endpoint"] == "openrouter":
        spend[f"eval: {label}"] = e["scores"]["all"]["cost_usd"]
for k, v in spend.items():
    print(f"{k:45} ${v:.4f}")
print(f"{'total':45} ${sum(spend.values()):.4f}")

teacher labels (1,350 tickets)                $0.1227
synthetic generation + QA pass                $0.0639
eval: gpt-6-luna, Ch. 2 prompt                $0.0199
eval: gpt-6-luna, + 22 examples               $0.0154
total                                         $0.2219
